In [15]:
import pandas as pd
import numpy as np
import torch as torch
import transformers
import esm
from torch.utils.data import DataLoader
import import_ipynb
import numpy as np
%run embedding_src.ipynb
from pathlib import Path

In [16]:
df = pd.read_parquet('data/processed/rsa_sequence_dataset.parquet')


In [17]:
rng = np.random.default_rng(42)
pdbs = df["pdb_id"].unique().to_numpy()
rng.shuffle(pdbs)

n = len(pdbs)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_pdbs = set(pdbs[:n_train])
val_pdbs   = set(pdbs[n_train:n_train + n_val])
test_pdbs  = set(pdbs[n_train + n_val:])

df_train = df[df["pdb_id"].isin(train_pdbs)].reset_index(drop=True)
df_val   = df[df["pdb_id"].isin(val_pdbs)].reset_index(drop=True)
df_test  = df[df["pdb_id"].isin(test_pdbs)].reset_index(drop=True)

In [18]:
train_ds = RSADataset(df_train)
val_ds   = RSADataset(df_val)
test_ds  = RSADataset(df_test)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_rsa)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_rsa)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=collate_rsa)

In [19]:
device = "mps"

model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

In [20]:
from tqdm import tqdm

In [21]:
cache_esm_embeddings(train_loader, model = model, batch_converter= batch_converter, device = device, cache_dir="cache/esm2_t30/train", layer=30)
cache_esm_embeddings(val_loader,  model = model, batch_converter= batch_converter, device = device, cache_dir="cache/esm2_t30/val",   layer=30)
cache_esm_embeddings(test_loader, model = model, batch_converter= batch_converter, device = device, cache_dir="cache/esm2_t30/test",  layer=30)

Caching ESM layer 30: 100%|███████████████████| 132/132 [03:08<00:00,  1.43s/it]
